In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd, numpy as np, json

ROOT=Path('/content/drive/MyDrive/US_ETF'); CACHE=ROOT/'directional_research/open_revalidation_1m_alpaca_v1/iex'; RES=ROOT/'model_lab_v1/results/open_revalidation_v1'; AUDIT=RES/'open_revalidation_trade_audit.parquet'
df=pd.read_parquet(AUDIT).copy(); NY=ZoneInfo('America/New_York')
fields=['entry_price_iex','fixed4_exit_price_iex','prev_close_price_iex','open_0_price_iex','open_5_price_iex','open_15_price_iex']
df['entry_timestamp']=pd.to_datetime(df['entry_timestamp'],utc=True,errors='coerce'); df['exit_timestamp']=pd.to_datetime(df['exit_timestamp'],utc=True,errors='coerce')
df['_symbol']=df['symbol'].astype(str).str.upper(); df['_entry_date_ny']=df['entry_timestamp'].dt.tz_convert(NY).dt.date.astype(str)
cached={p.name for p in CACHE.iterdir() if p.is_dir()}
df['_symbol_cache_missing']=~df['_symbol'].isin(cached)
df['_common5_missing']=df[['entry_price_iex','fixed4_exit_price_iex','open_0_price_iex','open_5_price_iex','open_15_price_iex']].isna().any(axis=1)
df['_prev_only_missing']=df['prev_close_price_iex'].isna() & ~df['_common5_missing']

print('='*100); print('KALMAN OPEN REVALIDATION MISSING DIAGNOSIS v1.7'); print('='*100)
print('rows=',len(df)); print('ready=',int(df[fields].notna().all(axis=1).sum()))
print('common5_missing_rows=',int(df['_common5_missing'].sum())); print('prev_close_missing_rows=',int(df['prev_close_price_iex'].isna().sum())); print('prev_close_only_missing_rows=',int(df['_prev_only_missing'].sum()))
print('symbol_cache_missing_rows=',int(df['_symbol_cache_missing'].sum()))
print('\n[MISSING CACHE SYMBOLS]')
print(df.loc[df['_symbol_cache_missing'],'_symbol'].value_counts().to_string())
print('\n[COMMON5 MISSING — TOP SYMBOLS]')
print(df.loc[df['_common5_missing'],'_symbol'].value_counts().head(30).to_string())
print('\n[PREV CLOSE ONLY MISSING — TOP SYMBOLS]')
print(df.loc[df['_prev_only_missing'],'_symbol'].value_counts().head(30).to_string())

# Build targeted queue. Common-5: entry date +/- 2d. Prev-close-only: extend 10 calendar days backward to capture prior trading session.
q=[]
for _,r in df.loc[df['_common5_missing'] | df['prev_close_price_iex'].isna()].iterrows():
    if pd.isna(r['entry_timestamp']): continue
    s=r['_symbol']; t=r['entry_timestamp']; start=t-pd.Timedelta(days=10 if pd.isna(r['prev_close_price_iex']) else 2); end=max(r['exit_timestamp'] if pd.notna(r['exit_timestamp']) else t,t)+pd.Timedelta(days=2)
    q.append({'symbol':s,'start':start.floor('D'),'end':end.ceil('D'),'reason':('COMMON5+' if r['_common5_missing'] else '')+('PREV_CLOSE' if pd.isna(r['prev_close_price_iex']) else '')})
q=pd.DataFrame(q)
if len(q):
    q=(q.groupby('symbol',as_index=False).agg(start=('start','min'),end=('end','max'),reason=('reason',lambda x:'|'.join(sorted(set(x))))))
    q['days']=(q['end']-q['start']).dt.days
out=RES/'open_revalidation_targeted_backfill_queue_v1_7.csv'; q.to_csv(out,index=False)
print('\n[TARGETED BACKFILL QUEUE] rows=',len(q)); print(q.to_string(index=False)); print('\nqueue=',out)
print('\nNOTE: This notebook is diagnostic/research-only. It does not modify the canonical audit or production/live trading.')
